# 41 — Capabilities, Monte-Carlo recovery (Group B)

Discrimination, guessing, covariates and latent correlation — each recovered over K datasets, probit and logit separate.

## Setup

In [ ]:
import sys
!{sys.executable} -m pip install -e "C:\\Users\\manonfi\\OneDrive - James Madison University\\Spring 26 classes & projects\\Practicum Python IRtree package\\PACKAGE\\Irtree"

In [ ]:
import sys, warnings, pathlib, importlib.util
import numpy as np, pandas as pd
np.set_printoptions(precision=4, suppress=True)
warnings.filterwarnings('ignore')

import irtree.tree_spec   as ts
import irtree.node_models as nm
import irtree.priors      as pr
import irtree.constraints as con
import irtree.ingest      as ing
import irtree.fit         as ft

# The data-generating code (simulate_irtree) lives in the test harness, NOT in
# the package: simulation is deliberately kept separate from fitting.  Loaded
# the same way notebooks 40 & 50 do.
ROOT = pathlib.Path.cwd().parents[1]
_hspec = importlib.util.spec_from_file_location("_harness", ROOT / "tests" / "_harness.py")
h = importlib.util.module_from_spec(_hspec); _hspec.loader.exec_module(h)

### Parameters

In [ ]:
# ---- design knobs: choose your own numbers ----
P, I          = 1500, 30
K             = 50          # simulated datasets per scenario (the main cost driver)
N_ITER_PROBIT = 4000        # probit = Albert-Chib Gibbs (mixes fast, cheaper)
N_ITER_LOGIT  = 8000        # logit  = Metropolis-Hastings (give it more)
N_CHAINS      = 2
SEED          = 20250413

# Truth is FIXED per scenario; only the response draw changes across the K
# datasets ("fix everything, redraw only responses").  Evidence-mode warmup is
# the package default, so each fit adapts its own warmup.

### Helpers

Fixed-truth Monte-Carlo machinery, shared by every scenario.

In [ ]:
def responses_frame(Y):
    """[P, I] response matrix -> keyed wide table for ingest.load_data."""
    Y = np.asarray(Y); Pn, In = Y.shape
    df = pd.DataFrame(Y, columns=[f'item{i+1}' for i in range(In)])
    df.insert(0, 'person_id', [f'p{p+1}' for p in range(Pn)])
    return df

def person_frame(X, names):
    X = np.asarray(X)
    df = pd.DataFrame(X, columns=list(names))
    df.insert(0, 'person_id', [f'p{p+1}' for p in range(X.shape[0])])
    return df

def item_frame(Q, names):
    Q = np.asarray(Q)
    df = pd.DataFrame(Q, columns=list(names))
    df.insert(0, 'item_id', [f'item{i+1}' for i in range(Q.shape[0])])
    return df


class MCStore:
    """Collects per-dataset recovery for a set of parameter-name PREFIXES.
    Truth is fixed; each dataset is one fresh response realization."""
    def __init__(self, truth_map):
        self.truth = {p: np.asarray(v, float).ravel() for p, v in truth_map.items()}
        self.post  = {p: [] for p in truth_map}
        self.lo    = {p: [] for p in truth_map}
        self.hi    = {p: [] for p in truth_map}
        self.tables   = []         # per-dataset DataFrame (all prefixes), kept for drill-down
        self.last_fit = None

    def add(self, fit):
        self.last_fit = fit
        names = fit.results[0].param_names
        draws = np.concatenate([np.asarray(r.chain, float) for r in fit.results])
        rowtabs = []
        for p, tv in self.truth.items():
            cols = [j for j, n in enumerate(names) if n.startswith(p)]
            if not cols:
                raise KeyError(f"no chain columns start with {p!r}; "
                               f"first names are {names[:8]}")
            d  = draws[:, cols]
            pm = d.mean(0); lo = np.percentile(d, 2.5, 0); hi = np.percentile(d, 97.5, 0)
            m  = min(len(tv), len(cols))
            self.post[p].append(pm[:m]); self.lo[p].append(lo[:m]); self.hi[p].append(hi[:m])
            rowtabs.append(pd.DataFrame({'param': [names[j] for j in cols][:m],
                                         'true': tv[:m], 'post_mean': pm[:m],
                                         'ci_lo': lo[:m], 'ci_hi': hi[:m]}))
        self.tables.append(pd.concat(rowtabs, ignore_index=True))

    def summary(self):
        rows = []
        for p, tv in self.truth.items():
            Pm = np.array(self.post[p]); Lo = np.array(self.lo[p]); Hi = np.array(self.hi[p])  # [K, n]
            m  = min(tv.shape[0], Pm.shape[1]); t = tv[:m]
            Pm, Lo, Hi = Pm[:, :m], Lo[:, :m], Hi[:, :m]
            err   = Pm - t[None, :]                                       # [K, n]
            cover = ((Lo <= t[None, :]) & (t[None, :] <= Hi)).mean(0)     # per element, over K
            rows.append(dict(param=p, n=m, K=Pm.shape[0],
                             mean_bias=float(err.mean()),
                             mean_abs_bias=float(np.abs(err.mean(0)).mean()),
                             rmse=float(np.sqrt((err ** 2).mean())),
                             coverage95=float(cover.mean())))
        return pd.DataFrame(rows)


def run_recovery(K, spec, gen_Y, make_data, make_models, n_iter, truth_map,
                 seed0, n_chains=None, make_priors=None):
    """Fix the truth; draw K response realizations; fit each; collect recovery."""
    n_chains = N_CHAINS if n_chains is None else n_chains
    store = MCStore(truth_map)
    for k in range(K):
        Y    = gen_Y(np.random.default_rng(seed0 + 1000 + k))   # response realization k
        data = make_data(Y)
        mods = make_models()
        priors = make_priors(mods) if make_priors is not None else None
        fit  = ft.irtree_fit(data, spec, mods, priors=priors,
                             n_iter=n_iter, n_chains=n_chains, seed=seed0 + k)
        store.add(fit)
    return store


def show_recovery(store, title):
    """Print the aggregate coverage/bias/RMSE table for one link."""
    print(title)
    s = store.summary()
    fmt = {c: '{:+.4f}'.format for c in ['mean_bias', 'mean_abs_bias', 'rmse']}
    fmt['coverage95'] = '{:.3f}'.format
    print(s.to_string(index=False, formatters=fmt))
    print()
    return s


def dataset_table(store, k):
    """Per-dataset posterior-mean-vs-truth table for dataset k (drill-down)."""
    t = store.tables[k].copy()
    t['abs_err'] = (t['post_mean'] - t['true']).abs()
    print(f'dataset {k}:')
    fmt = {c: '{:+.3f}'.format for c in ['true', 'post_mean', 'ci_lo', 'ci_hi', 'abs_err']}
    print(t.to_string(index=False, formatters=fmt))
    return t

### How to read this notebook

Each scenario fits **K independently simulated datasets** that share one fixed truth (only the response draw changes). Probit and logit are separate cells so you can give each its own iteration count (`N_ITER_PROBIT`, `N_ITER_LOGIT`).

Each link prints one aggregate table: per parameter group, `mean_bias`, `mean_abs_bias`, `rmse`, and `coverage95` (the fraction of the K fits whose 95% interval contains the truth; well-calibrated ≈ 0.95). The per-dataset tables are retained — call `dataset_table(store, k)` on any scenario’s store (e.g. `A1p`, `A1l`) to see dataset k in full.

## B1. Discrimination (2PL) at the ability node

**Simulate** (fixed truth; only responses redrawn across K)

In [ ]:
# B1: node-2 discrimination (2PL)
spec_cur = ts.disengagement_tree(binary=True); seed_scn = SEED + 10
rng = np.random.default_rng(seed_scn)
theta1, theta2 = rng.normal(0,1,P), rng.normal(0,1,P)
b1 = rng.normal(0.6,0.7,I); b2 = rng.normal(0,1,I)
a2 = rng.uniform(0.7, 1.6, I)
truth = [dict(theta=theta1, b=b1), dict(theta=theta2, b=b2, a=a2)]
truth_map = {'b_1_': b1, 'b_2_': b2, 'a_2_': a2}
def gen_pro(r): return h.simulate_irtree(spec_cur, truth, P, I, r, node_links=['probit','probit'])
def gen_log(r): return h.simulate_irtree(spec_cur, truth, P, I, r, node_links=['logit','logit'])
def load_cur(Y): return ing.load_data(responses_frame(Y), spec=spec_cur, categories_start=1)
make_pro = lambda: [nm.ProbitRaschNode(), nm.ProbitTwoPLNode()]
make_log = lambda: [nm.LogitRaschNode(),  nm.LogitTwoPLNode()]

**Probit** — K datasets, `N_ITER_PROBIT`

In [ ]:
B1p = run_recovery(K, spec_cur, gen_pro, load_cur, make_pro, N_ITER_PROBIT, truth_map, seed_scn)
show_recovery(B1p, 'B1 PROBIT  (2PL)')

**Logit** — K datasets, `N_ITER_LOGIT`

In [ ]:
B1l = run_recovery(K, spec_cur, gen_log, load_cur, make_log, N_ITER_LOGIT, truth_map, seed_scn)
show_recovery(B1l, 'B1 LOGIT  (2PL)')
print('MH froze at (last fit):', B1l.last_fit.results[0].frozen_at)

## B2. Guessing (3PL) at the ability node

**Simulate** (fixed truth; only responses redrawn across K)

In [ ]:
# B2: node-2 discrimination + guessing (3PL)
spec_cur = ts.disengagement_tree(binary=True); seed_scn = SEED + 11
rng = np.random.default_rng(seed_scn)
theta1, theta2 = rng.normal(0,1,P), rng.normal(0,1,P)
b1 = rng.normal(0.6,0.7,I); b2 = rng.normal(0,1,I)
a2 = rng.uniform(0.8,1.5,I); c2 = np.full(I, 0.2)
truth = [dict(theta=theta1, b=b1), dict(theta=theta2, b=b2, a=a2, c=c2)]
truth_map = {'b_1_': b1, 'b_2_': b2, 'a_2_': a2, 'c_2_': c2}
def gen_pro(r): return h.simulate_irtree(spec_cur, truth, P, I, r, node_links=['probit','probit'])
def gen_log(r): return h.simulate_irtree(spec_cur, truth, P, I, r, node_links=['logit','logit'])
def load_cur(Y): return ing.load_data(responses_frame(Y), spec=spec_cur, categories_start=1)
make_pro = lambda: [nm.ProbitRaschNode(), nm.ProbitThreePLNode()]
make_log = lambda: [nm.LogitRaschNode(),  nm.LogitThreePLNode()]

**Probit** — K datasets, `N_ITER_PROBIT`

In [ ]:
B2p = run_recovery(K, spec_cur, gen_pro, load_cur, make_pro, N_ITER_PROBIT, truth_map, seed_scn)
show_recovery(B2p, 'B2 PROBIT  (3PL)')

**Logit** — K datasets, `N_ITER_LOGIT`

In [ ]:
B2l = run_recovery(K, spec_cur, gen_log, load_cur, make_log, N_ITER_LOGIT, truth_map, seed_scn)
show_recovery(B2l, 'B2 LOGIT  (3PL)')

`c` (guessing) is weakly identified; expect wider intervals and lower coverage than `a`/`b`.

## B3. MH diagnostics (logit only)

The one scenario with no probit twin: Metropolis-Hastings only drives logit nodes, so there is nothing to split. Watch acceptance and the evidence-warmup freeze point alongside `b` recovery.

**Simulate** (fixed truth; only responses redrawn across K)

In [ ]:
# B3: MH diagnostics (logit only -- MH exists only for logit nodes)
spec_cur = ts.disengagement_tree(binary=True); seed_scn = SEED + 12
rng = np.random.default_rng(seed_scn)
theta1, theta2 = rng.normal(0,1,P), rng.normal(0,1,P)
b1 = rng.normal(0.6,0.7,I); b2 = rng.normal(0,1,I)
truth = [dict(theta=theta1, b=b1), dict(theta=theta2, b=b2)]
truth_map = {'b_1_': b1, 'b_2_': b2}
def gen_log(r): return h.simulate_irtree(spec_cur, truth, P, I, r, node_links=['logit','logit'])
def load_cur(Y): return ing.load_data(responses_frame(Y), spec=spec_cur, categories_start=1)
make_log = lambda: [nm.LogitRaschNode(), nm.LogitRaschNode()]

**Logit** — K datasets, `N_ITER_LOGIT`

In [ ]:
B3l = run_recovery(K, spec_cur, gen_log, load_cur, make_log, N_ITER_LOGIT, truth_map, seed_scn)
show_recovery(B3l, 'B3 LOGIT  (MH diagnostics)')
# evidence-mode warmup: where each MH block froze, across the K fits
froze = [f.results[0].frozen_at for f in [B3l.last_fit]]
print('accept rates (last fit):', {k: round(v,3) for k,v in B3l.last_fit.results[0].accept_rates.items()})
print('froze at     (last fit):', B3l.last_fit.results[0].frozen_at)

## B4. Person covariate on the disengagement latent

**Simulate** (fixed truth; only responses redrawn across K)

In [ ]:
# B4: person covariate on the node-1 latent (regression on disengagement)
spec_cur = ts.disengagement_tree(binary=True); seed_scn = SEED + 13
rng = np.random.default_rng(seed_scn)
Xmat = rng.normal(size=(P,2)); Xmat -= Xmat.mean(0)       # centre covariates
beta_true = np.array([0.5, -0.4])
theta1 = rng.normal(0,1,P) + Xmat @ beta_true
theta2 = rng.normal(0,1,P)
b1 = rng.normal(0.6,0.7,I); b2 = rng.normal(0,1,I)
truth = [dict(theta=theta1, b=b1), dict(theta=theta2, b=b2)]
truth_map = {'b_1_': b1, 'b_2_': b2, 'beta_1_': beta_true}
xdf = person_frame(Xmat, ['escs','female'])
def gen_pro(r): return h.simulate_irtree(spec_cur, truth, P, I, r, node_links=['probit','probit'])
def gen_log(r): return h.simulate_irtree(spec_cur, truth, P, I, r, node_links=['logit','logit'])
def load_cur(Y): return ing.load_data(responses_frame(Y), spec=spec_cur, person_covariates=xdf, categories_start=1)
make_pro = lambda: [nm.ProbitRaschNode(person_covariates=['escs','female']), nm.ProbitRaschNode()]
make_log = lambda: [nm.LogitRaschNode(person_covariates=['escs','female']),  nm.LogitRaschNode()]

**Probit** — K datasets, `N_ITER_PROBIT`

In [ ]:
B4p = run_recovery(K, spec_cur, gen_pro, load_cur, make_pro, N_ITER_PROBIT, truth_map, seed_scn)
show_recovery(B4p, 'B4 PROBIT  (person covariate)')

**Logit** — K datasets, `N_ITER_LOGIT`

In [ ]:
B4l = run_recovery(K, spec_cur, gen_log, load_cur, make_log, N_ITER_LOGIT, truth_map, seed_scn)
show_recovery(B4l, 'B4 LOGIT  (person covariate)')

## B5. Latent correlation ρ between node latents

**Simulate** (fixed truth; only responses redrawn across K)

In [ ]:
# B5: latent correlation between the two node latents
spec_cur = ts.disengagement_tree(binary=True); seed_scn = SEED + 14
rng = np.random.default_rng(seed_scn)
rho_true = -0.5
L  = np.linalg.cholesky(np.array([[1, rho_true],[rho_true, 1]]))
TH = rng.standard_normal((P,2)) @ L.T
b1 = rng.normal(0.6,0.7,I); b2 = rng.normal(0,1,I)
truth = [dict(theta=TH[:,0], b=b1), dict(theta=TH[:,1], b=b2)]
truth_map = {'rho_': np.array([rho_true]), 'b_1_': b1, 'b_2_': b2}
def gen_pro(r): return h.simulate_irtree(spec_cur, truth, P, I, r, node_links=['probit','probit'])
def gen_log(r): return h.simulate_irtree(spec_cur, truth, P, I, r, node_links=['logit','logit'])
def load_cur(Y): return ing.load_data(responses_frame(Y), spec=spec_cur, categories_start=1)
make_pro = lambda: [nm.ProbitRaschNode(), nm.ProbitRaschNode()]
make_log = lambda: [nm.LogitRaschNode(),  nm.LogitRaschNode()]
def mk_lkj(mods): 
    p = pr.build_default_priors(spec_cur, mods)
    p.type_level['theta.correlation'] = pr.LKJPrior(eta=1.0)
    return p

**Probit** — K datasets, `N_ITER_PROBIT`

In [ ]:
B5p = run_recovery(K, spec_cur, gen_pro, load_cur, make_pro, N_ITER_PROBIT, truth_map, seed_scn, make_priors=mk_lkj)
show_recovery(B5p, 'B5 PROBIT  (latent correlation)')

**Logit** — K datasets, `N_ITER_LOGIT`

In [ ]:
B5l = run_recovery(K, spec_cur, gen_log, load_cur, make_log, N_ITER_LOGIT, truth_map, seed_scn, make_priors=mk_lkj)
show_recovery(B5l, 'B5 LOGIT  (latent correlation)')

`rho_` is the single `rho_1_2` column; coverage here is over the K datasets for that one scalar.